# Comparaison des méthodes d'optimisation — HEAT-COND

Ce notebook compare quantitativement les **trois optimiseurs** implémentés dans
`src/optimization.py` sur le problème HEAT-COND :

- **Differential Evolution** (DE) — global, stochastique, population.
- **Nelder-Mead** (NM) — local, déterministe, sans gradient.
- **Basinhopping** (BH) — global, hybride (perturbation + L-BFGS-B local).

Métriques évaluées :

- Meilleur $J^\star$ atteint et gain absolu/relatif vs $J_0$ (design uniforme $x = 0.5$).
- Nombre d'évaluations PDE et temps mur.
- Vitesse de convergence (running best vs n_eval).
- Trade-off coût/qualité (J* vs temps).
- Robustesse multi-graines (DE, BH) / multi-points initiaux (NM).
- Comparaison des designs optimaux et des champs $T$.

**Pré-requis** : FreeFEM++ installé, `.env` configuré, exécuter depuis `heat_opti_final/`.
Les résultats du notebook sont écrits dans `results_compare/`.


## 0. Configuration


In [ ]:
import os, sys, time
from pathlib import Path

# Permettre les imports src.* depuis le notebook lancé dans heat_opti_final/
ROOT = Path.cwd()
if not (ROOT / 'src' / 'optimization.py').exists():
    # On a peut-être lancé depuis un parent — ajustement
    candidate = ROOT / 'heat_opti_final'
    if (candidate / 'src' / 'optimization.py').exists():
        os.chdir(candidate)
        ROOT = candidate
sys.path.insert(0, str(ROOT))
print('Working directory :', ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.freefem_interface import ensure_mesh, run_solver, read_temperature_field
from src.optimization import (
    run_differential_evolution, run_nelder_mead, run_basinhopping,
    reset_optimization,
)
from src.visualization import draw_temperature, draw_convergence_comparison

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

BOUNDS = [(0.1, 1.0)] * 5 + [(0.01, 1.0)]
PARAM_LABELS = ['k1', 'k2', 'k3', 'k4', 'k5', 'Bi']
MESH_SIZE = 50  # référence pour les comparaisons

NB_DIR = ROOT / 'results_compare'
NB_DIR.mkdir(exist_ok=True)
print(f'Sorties du notebook  : {NB_DIR}')


## 1. Design initial (référence)

$J_0$ pour $x_0 = (0.5, 0.5, 0.5, 0.5, 0.5, 0.5)$. Sert de baseline pour tous les gains.


In [ ]:
ensure_mesh(MESH_SIZE)  # mise en cache
x0_ref = [0.5] * 5 + [0.5]
t = time.time()
J0 = run_solver(x0_ref, mesh_size=MESH_SIZE)
print(f'J0 = {J0:.8f}   ({time.time()-t:.2f} s pour 1 solve)')


## 2. Exécution des trois méthodes

Budget volontairement modeste pour que le notebook tourne en 1-3 min selon ta machine.
Pour le rapport, augmente `maxiter`/`niter` si tu veux des courbes plus lisses.

| Méthode | Hyperparamètres | Budget approx. |
|---|---|---|
| Differential Evolution | `maxiter=10, popsize=6` | ~70 évaluations |
| Nelder-Mead            | `maxiter=80`, x0 = milieu des bornes | 80-150 éval. |
| Basinhopping           | `niter=15`, local = L-BFGS-B | variable, souvent élevé |


In [ ]:
results = {}

print('=== Differential Evolution ===')
results['DE'] = run_differential_evolution(
    BOUNDS, maxiter=10, popsize=6, mesh_size=MESH_SIZE, seed=42,
)
reset_optimization()

print('\n=== Nelder-Mead ===')
results['NM'] = run_nelder_mead(BOUNDS, maxiter=80, mesh_size=MESH_SIZE)
reset_optimization()

print('\n=== Basinhopping ===')
results['BH'] = run_basinhopping(BOUNDS, niter=15, mesh_size=MESH_SIZE, seed=42)
reset_optimization()


## 3. Tableau de synthèse

Toutes les métriques dans un seul tableau (sauvegardé en CSV pour le rapport).


In [ ]:
rows = []
for key, r in results.items():
    rows.append({
        'Méthode'          : r['method'],
        'J*'               : r['best_J'],
        'Gain vs J0'       : r['best_J'] - J0,
        'Gain relatif (%)' : 100 * (r['best_J'] - J0) / abs(J0),
        'n_eval'           : r['n_eval'],
        'Temps (s)'        : r['time'],
        'Temps/éval (s)'   : r['time'] / max(1, r['n_eval']),
        'Convergé'         : bool(r['success']),
    })
df_summary = pd.DataFrame(rows).sort_values('J*', ascending=False)
df_summary.to_csv(NB_DIR / 'summary.csv', index=False)
df_summary.style.format({
    'J*': '{:.6f}', 'Gain vs J0': '{:+.6f}', 'Gain relatif (%)': '{:+.2f}',
    'Temps (s)': '{:.2f}', 'Temps/éval (s)': '{:.3f}',
})


## 4. Convergence comparée

Meilleur $J$ atteint en fonction du nombre d'évaluations PDE. Plus la courbe monte
vite, plus la méthode est efficiente en termes d'appels au solveur.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
draw_convergence_comparison(
    ax,
    {r['method']: r['history'] for r in results.values()},
)
ax.axhline(J0, color='gray', ls='--', lw=1.2, label=f'J0 = {J0:.4f}')
ax.legend()
fig.tight_layout()
fig.savefig(NB_DIR / 'convergence_compare.png', dpi=200, bbox_inches='tight')
plt.show()


## 5. Trade-off coût / qualité — *Pareto* J\* vs temps

Une méthode est dominée si une autre fait mieux (J plus grand) ET plus vite.
Visuellement, on cherche le coin haut-gauche.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.tab10.colors
for i, (key, r) in enumerate(results.items()):
    ax.scatter(r['time'], r['best_J'], s=160, color=colors[i],
               label=r['method'], edgecolors='black', linewidths=1)
    ax.annotate(r['method'], (r['time'], r['best_J']),
                xytext=(8, 6), textcoords='offset points', fontsize=10)
ax.axhline(J0, color='gray', ls='--', lw=1, label=f'J0 = {J0:.4f}')
ax.set_xlabel('Temps de calcul (s)')
ax.set_ylabel('Meilleur J atteint')
ax.set_title('Performance : meilleur J vs temps')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(NB_DIR / 'pareto.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Robustesse (multi-graines / multi-points initiaux)

On évalue la variabilité du $J^\star$ final :

- DE et BH : 5 graines différentes (méthodes stochastiques).
- NM : 5 points initiaux tirés uniformément dans les bornes (méthode locale déterministe).

Budget réduit pour que ça tourne ; agrandir si besoin pour le rapport.


In [ ]:
n_runs = 5
robust = {'DE': [], 'NM': [], 'BH': []}
rng = np.random.default_rng(0)

print('--- DE multi-seed ---')
for s in range(n_runs):
    r = run_differential_evolution(BOUNDS, maxiter=5, popsize=4,
                                    mesh_size=MESH_SIZE, seed=s)
    robust['DE'].append(r['best_J'])
    reset_optimization()
    print(f'  seed={s}: J* = {r["best_J"]:.6f}')

print('\n--- BH multi-seed ---')
for s in range(n_runs):
    r = run_basinhopping(BOUNDS, niter=8, mesh_size=MESH_SIZE, seed=s)
    robust['BH'].append(r['best_J'])
    reset_optimization()
    print(f'  seed={s}: J* = {r["best_J"]:.6f}')

print('\n--- NM multi-x0 ---')
for k in range(n_runs):
    x0 = rng.uniform([b[0] for b in BOUNDS], [b[1] for b in BOUNDS])
    r = run_nelder_mead(BOUNDS, x0=x0, maxiter=40, mesh_size=MESH_SIZE)
    robust['NM'].append(r['best_J'])
    reset_optimization()
    print(f'  x0[{k}]: J* = {r["best_J"]:.6f}')

df_robust = pd.DataFrame(robust)
df_robust.describe().T.to_csv(NB_DIR / 'robustness.csv')
df_robust.describe()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(
    [robust['DE'], robust['NM'], robust['BH']],
    tick_labels=['DE (5 graines)', 'NM (5 x0)', 'BH (5 graines)'],
    patch_artist=True,
)
for patch, c in zip(bp['boxes'], plt.cm.tab10.colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.axhline(J0, color='gray', ls='--', lw=1, label=f'J0 = {J0:.4f}')
ax.set_ylabel('J* final')
ax.set_title(f'Variabilité du J* sur {n_runs} runs')
ax.legend()
fig.tight_layout()
fig.savefig(NB_DIR / 'robustness_box.png', dpi=200, bbox_inches='tight')
plt.show()


## 7. Comparaison des designs optimaux

Les trois méthodes convergent-elles vers le même optimum ? Si elles divergent
fortement, c'est le signe de plusieurs optima locaux (ou d'un budget trop court).


In [ ]:
designs = pd.DataFrame(
    {r['method']: list(r['best_x']) for r in results.values()},
    index=PARAM_LABELS,
).T
designs['J*'] = [r['best_J'] for r in results.values()]
designs.to_csv(NB_DIR / 'designs.csv')
designs.style.format('{:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(6)
w = 0.25
colors = plt.cm.tab10.colors
for i, (key, r) in enumerate(results.items()):
    ax.bar(x + i * w, list(r['best_x']), w,
           label=r['method'], color=colors[i], edgecolor='black', linewidth=0.5)
ax.set_xticks(x + w)
ax.set_xticklabels(PARAM_LABELS)
ax.set_ylabel('Valeur du paramètre')
ax.set_title('Designs optimaux par méthode')
ax.legend()
fig.tight_layout()
fig.savefig(NB_DIR / 'designs_bar.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Champs de température

Pour chaque $x^\star$, on recalcule le champ $T$ et on l'affiche avec l'échelle
de couleurs partagée. Interprétation physique : à conductivité d'ailette optimisée,
$T$ devrait être plus élevé sur les ailettes (= plus de transfert thermique vers l'extérieur).


In [ ]:
all_data = {}

T_init_path = NB_DIR / 'T_initial.dat'
run_solver(x0_ref, mesh_size=MESH_SIZE, t_out=str(T_init_path))
all_data['Initial (x = 0.5)'] = read_temperature_field(T_init_path)

for key, r in results.items():
    T_path = NB_DIR / f'T_{key}.dat'
    run_solver(list(r['best_x']), mesh_size=MESH_SIZE, t_out=str(T_path))
    all_data[r['method']] = read_temperature_field(T_path)

vmin = min(d[2].min() for d in all_data.values())
vmax = max(d[2].max() for d in all_data.values())

n = len(all_data)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5))
tcf = None
for ax, (name, (x, y, T)) in zip(axes, all_data.items()):
    tcf = draw_temperature(ax, x, y, T, title=name, vmin=vmin, vmax=vmax)
fig.colorbar(tcf, ax=axes, fraction=0.025, pad=0.04, label='T')
fig.suptitle('Champs T : design initial et meilleurs designs par méthode')
fig.savefig(NB_DIR / 'T_fields_compare.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Conclusion (à reprendre dans le rapport)

Quelques observations typiques à confirmer/réfuter en regardant les figures ci-dessus :

- **DE** est globalement le meilleur en qualité finale, mais paie le coût (popsize × maxiter évaluations).
- **Nelder-Mead** est rapide mais sensible au point initial (cf. boxplot section 6).
- **Basinhopping** explore bien grâce aux sauts mais le coût total est souvent supérieur à DE.
- Les designs $x^\star$ tendent à converger vers des $k_i$ élevés (favoriser la conduction des ailettes vers l'extérieur)
  et un $\mathrm{Bi}$ faible (limiter la perte radiative).

**Pour le rapport** :

- Section *Convergence behavior* → figure 4 + tableau 3.
- Section *Computational cost* → figure 5 + colonne `Temps/éval`.
- Section *Sensitivity* → figure 6 (boxplot) + tableau de `df_robust.describe()`.
- Section *Physical interpretation* → figure 8 + tableau 7.
